# OIStats-v3: Lab 1 — Introduction to Data with Python

**Name:** ______________________________

This notebook is a Python/Jupyter adaptation of the OpenIntro *Introduction to Data* lab.

Statistics can be viewed as the process of turning information into knowledge. A first step is to summarize and describe raw data. In this lab, you will gain insight into public health by generating graphical and numerical summaries of a data set collected by the Centers for Disease Control and Prevention (CDC). Along the way, you will practice data processing and subsetting with pandas.

## Getting started

The Behavioral Risk Factor Surveillance System (BRFSS) is a large U.S. health survey. We will work with a random sample of 20,000 respondents from the 2000 survey and a subset of nine variables.

Run the imports and then load the CSV data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)

### Local course data

The BRFSS data are supplied as `./data/cdc.csv`. pandas can read the CSV file
directly into a DataFrame with `pd.read_csv()`.

The loading code below also checks the file location and verifies that the
columns needed for this lab are present.

In [ ]:
from pathlib import Path

DATA_DIR = Path("./data")
CDC_PATH = DATA_DIR / "cdc.csv"

if not CDC_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CDC_PATH.resolve()}. "
        "Place cdc.csv in a data folder in the notebook's working directory."
    )

In [ ]:
cdc = pd.read_csv(
    CDC_PATH,
    na_values=["?", "NA", "N/A", ""],
    skipinitialspace=True,
)

# Remove an accidental saved-index column, if the CSV was exported with one.
cdc.columns = cdc.columns.str.strip()
cdc = cdc.loc[:, ~cdc.columns.str.match(r"^Unnamed")].copy()

required_columns = [
    "genhlth", "exerany", "hlthplan", "smoke100", "height",
    "weight", "wtdesire", "age", "gender",
]
missing_columns = [column for column in required_columns if column not in cdc.columns]
if missing_columns:
    raise ValueError(
        "cdc.csv is missing required columns: " + ", ".join(missing_columns)
    )

# Clean categorical text so later comparisons such as gender == "m" are reliable.
for column in ["genhlth", "gender"]:
    cdc[column] = cdc[column].astype("string").str.strip().str.lower()

# Keep the general-health categories in their natural order for grouped plots.
health_order = ["excellent", "very good", "good", "fair", "poor"]
observed_health = set(cdc["genhlth"].dropna().unique())
if observed_health.issubset(health_order):
    cdc["genhlth"] = pd.Categorical(
        cdc["genhlth"], categories=health_order, ordered=True
    )

cdc.head()

Each row is a **case/observation** and each column is a **variable**.

The variables are:

- `genhlth`: general health (`excellent`, `very good`, `good`, `fair`, `poor`)
- `exerany`: exercised in the past month (1 = yes, 0 = no)
- `hlthplan`: has some form of health coverage (1 = yes, 0 = no)
- `smoke100`: smoked at least 100 cigarettes in their lifetime (1 = yes, 0 = no)
- `height`: height in inches
- `weight`: weight in pounds
- `wtdesire`: desired weight in pounds
- `age`: age in years
- `gender`: gender in the source data

In [ ]:
cdc.columns

### Exercise 1

How many cases are in this data set? How many variables? For each variable, identify its data type in a statistical sense (for example, categorical or quantitative/discrete).

In [ ]:
# Exercise 1
# Helpful Python:
cdc.shape

You can inspect the first or last several rows with `.head()` and `.tail()`.

In [ ]:
cdc.head()

In [ ]:
cdc.tail()

## Summaries and tables

A good first step in analysis is often to summarize a variable. pandas `.describe()` reports the count, mean, standard deviation, minimum, quartiles, and maximum for a quantitative Series.

In [ ]:
cdc["weight"].describe()

The interquartile range (IQR) is $Q_3-Q_1$.

In [ ]:
q1 = cdc["weight"].quantile(0.25)
q3 = cdc["weight"].quantile(0.75)
iqr_weight = q3 - q1

q1, q3, iqr_weight

pandas also provides separate methods for the mean, sample variance, and median.

In [ ]:
cdc["weight"].mean(), cdc["weight"].var(), cdc["weight"].median()

For categorical data, frequency and relative-frequency tables are more appropriate. Use `.value_counts()`.

In [ ]:
cdc["smoke100"].value_counts().sort_index()

In [ ]:
cdc["smoke100"].value_counts(normalize=True).sort_index()

The frequency table can be plotted directly.

In [ ]:
smoke = cdc["smoke100"].value_counts().sort_index()

smoke.plot(kind="bar")
plt.xlabel("Smoked at least 100 cigarettes")
plt.ylabel("Count")
plt.title("Smoking history")
plt.show()

### Exercise 2

1. Create numerical summaries for `height` and `age`, and compute the IQR for each.
2. Compute the relative-frequency distributions for `gender` and `exerany`.
3. How many males are in the sample?
4. What proportion of the sample reports being in excellent health?

In [ ]:
# Exercise 2
# Your code here:

## Two-way tables

`pd.crosstab()` is the pandas equivalent of tabulating two categorical variables together.

In [ ]:
smoke_by_gender = pd.crosstab(cdc["gender"], cdc["smoke100"])
smoke_by_gender

The original R lab used a mosaic plot. We can create one in Python with `statsmodels`.

In [ ]:
from statsmodels.graphics.mosaicplot import mosaic

mosaic(cdc, ["gender", "smoke100"])
plt.title("Smoking history by gender")
plt.show()

### Exercise 3

What does the mosaic plot reveal about smoking habits and gender?

**Answer:**

## How pandas thinks about data

A DataFrame is a rectangular table. pandas supports both **position-based** indexing with `.iloc[]` and **label-based** indexing with `.loc[]`.

Python uses **zero-based indexing**. Therefore, the 567th respondent is at position 566.

To inspect the sixth variable of the 567th respondent:

In [ ]:
cdc.iloc[566, 5]

To see the sixth variable for the first 10 respondents:

In [ ]:
cdc.iloc[:10, 5]

To see all variables for the first 10 respondents:

In [ ]:
cdc.iloc[:10, :]

Selecting by variable name is usually clearer:

In [ ]:
cdc["weight"]

The 567th respondent's weight:

In [ ]:
cdc["weight"].iloc[566]

The first 10 weights:

In [ ]:
cdc["weight"].iloc[:10]

## A little more on subsetting

Boolean conditions produce a Series of `True` and `False` values.

In [ ]:
cdc["gender"] == "m"

In [ ]:
cdc["age"] > 30

Use a Boolean mask inside `.loc[]` to select rows.

In [ ]:
mdata = cdc.loc[cdc["gender"] == "m"].copy()
mdata.head()

Multiple conditions use `&` for **and** and `|` for **or**. Put each comparison in parentheses.

In [ ]:
m_and_over30 = cdc.loc[(cdc["gender"] == "m") & (cdc["age"] > 30)].copy()
m_and_over30.head()

In [ ]:
m_or_over30 = cdc.loc[(cdc["gender"] == "m") | (cdc["age"] > 30)].copy()
m_or_over30.head()

### Exercise 4

Create a new DataFrame called `under23_and_smoke` containing all respondents who are **under age 23** and **have smoked at least 100 cigarettes in their lifetime**. Write and run the command below.

In [ ]:
# Exercise 4
# under23_and_smoke = ...

## Quantitative data: box plots

Two common ways to visualize a quantitative variable are box plots and histograms.

In [ ]:
plt.boxplot(cdc["height"], vert=True)
plt.ylabel("Height (inches)")
plt.title("Height")
plt.show()

In [ ]:
cdc["height"].describe()

To compare height across genders:

In [ ]:
cdc.boxplot(column="height", by="gender")
plt.suptitle("")
plt.title("Height by gender")
plt.xlabel("Gender")
plt.ylabel("Height (inches)")
plt.show()

## Body Mass Index

Define BMI using

$$
\mathrm{BMI}=\frac{\mathrm{weight\ (lb)}}{\mathrm{height\ (in)}^2}\times 703.
$$

Vectorized arithmetic lets us compute BMI for all 20,000 respondents at once.

In [ ]:
cdc["bmi"] = (cdc["weight"] / cdc["height"]**2) * 703
cdc[["height", "weight", "bmi"]].head()

In [ ]:
cdc.boxplot(column="bmi", by="genhlth")
plt.suptitle("")
plt.title("BMI by self-reported general health")
plt.xlabel("General health")
plt.ylabel("BMI")
plt.xticks(rotation=30)
plt.show()

### Exercise 5

What does this box plot show? Pick another categorical variable from the data set and examine how it relates to BMI. State:

1. the variable you chose,
2. why you might expect it to be related to BMI, and
3. what the figure appears to suggest.

In [ ]:
# Exercise 5
# Your plot/code here:

**Answer:**

## Histograms

Histograms show the shape of a quantitative distribution.

In [ ]:
plt.hist(cdc["age"].dropna())
plt.xlabel("Age")
plt.ylabel("Count")
plt.title("Age distribution")
plt.show()

The number of bins can change the appearance of a histogram.

In [ ]:
plt.hist(cdc["bmi"].dropna())
plt.xlabel("BMI")
plt.ylabel("Count")
plt.title("BMI distribution — default bins")
plt.show()

In [ ]:
plt.hist(cdc["bmi"].dropna(), bins=50)
plt.xlabel("BMI")
plt.ylabel("Count")
plt.title("BMI distribution — 50 bins")
plt.show()

How do the two BMI histograms compare?

**Answer:**

# On Your Own

Use Python to answer the following questions.

1. Make a scatterplot of `weight` versus `wtdesire`. Describe the relationship.
2. Create `wdiff = wtdesire - weight`.
3. What type of data is `wdiff`? What does a value of 0 mean? What do positive and negative values mean?
4. Describe the distribution of `wdiff` in terms of center, shape, and spread, including any plots you use.
5. Using numerical summaries and a side-by-side box plot, determine whether men tend to view their weight differently than women.
6. Find the mean and standard deviation of weight and determine what proportion of weights are within one standard deviation of the mean.
7. What concepts from the textbook are covered in this lab? What concepts, if any, are not covered in the textbook? Where else have you encountered them?

In [ ]:
# On Your Own 1

In [ ]:
# On Your Own 2–4

In [ ]:
# On Your Own 5

In [ ]:
# On Your Own 6

### On Your Own 7 — Written response

## Attribution

This lab is adapted from an OpenIntro lab released under the **Creative Commons Attribution-ShareAlike 3.0 Unported License**. The original lab was adapted for OpenIntro by Andrew Bray and Mine Çetinkaya-Rundel from a lab written by Mark Hansen of UCLA Statistics.

Python/Jupyter adaptation for course use, 2026.